In [1]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from fractions import Fraction

In [2]:
import pandas as pd
df = pd.read_csv('../output/Bach_chordify_int_5.csv')
display(df.head())
df.drop(['file', 'position'], axis=1, inplace=True)

,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5,label,duration,file,position
0,0,0,0,0,0,16,0.25,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",0
1,0,0,0,0,16,528,0.25,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",1
2,0,0,0,16,528,656,0.25,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",2
3,0,0,16,528,656,128,0.50,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",3
4,0,16,528,656,128,130,0.50,"Bach, Carl Philipp Emanuel, Keyboard Sonata in...",4


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
import pandas as pd
from sklearn.preprocessing import LabelEncoder

lookback_cols = [c for c in df.columns if 'lookback' in c and df[c].dtype == 'object' or 'label' in c]

all_chords = set(df['label'].unique())
for col in lookback_cols:
    all_chords |= set(df[col].dropna().unique())

le_chord = LabelEncoder()
le_chord.fit(sorted(all_chords))

for col in lookback_cols:
    df[col] = le_chord.transform(df[col])


display(df.head())

,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5,label,duration
0,0,0,0,0,0,13,0.25
1,0,0,0,0,16,226,0.25
2,0,0,0,16,528,291,0.25
3,0,0,16,528,656,74,0.50
4,0,16,528,656,128,76,0.50


In [4]:
X_chord = df.drop(columns=['label', 'duration'])
y_chord = df['label']
X_duration = df.drop(columns=['duration'])
df['duration'] = df['duration'].apply(lambda x: Fraction(x).limit_denominator())
from sklearn.preprocessing import LabelEncoder
le_dur = LabelEncoder()
le_dur.fit(sorted(df['duration'].unique()))
df['duration'] = le_dur.transform(df['duration'])
y_duration = df['duration']
print("Training samples:", X_chord.shape[0])
print("Feature dims:",    X_chord.shape[1])
display(X_chord.head())
display(y_duration.head())

Training samples: 200525
Feature dims: 5


,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5
0,0,0,0,0,0
1,0,0,0,0,16
2,0,0,0,16,528
3,0,0,16,528,656
4,0,16,528,656,128


0    2
1    2
2    2
3    5
4    5
Name: duration, dtype: int64

In [5]:
n_samples = X_chord.shape[0]
n_features = X_chord.shape[1]
print(f"Ratio samples/features = {n_samples/n_features:.1f}")

Ratio samples/features = 40105.0


In [11]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import learning_curve, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import RBFSampler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import learning_curve
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_chord, y_chord, test_size=0.2, random_state=42)
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_duration, y_duration, test_size=0.2, random_state=42)
display(Xc_train.head())

,lookback_1,lookback_2,lookback_3,lookback_4,lookback_5
151994,144,4240,144,36,526336
104651,4804,128,128,128,2176
148815,149632,18560,2176,6784,6784
31761,129,129,1,1,4
146666,18432,512,17024,16512,32


In [7]:
from sklearn.metrics import accuracy_score
clf_chord = DecisionTreeClassifier(random_state=42)
clf_duration = DecisionTreeClassifier(random_state=42)
clf_chord.fit(Xc_train, yc_train)
clf_duration.fit(Xd_train, yd_train)
yc_pred = clf_chord.predict(Xc_test)
yd_pred = clf_duration.predict(Xd_test)
print("Test accuracy (chord):", accuracy_score(yc_test, yc_pred))
print("Test accuracy (duration):", accuracy_score(yd_test, yd_pred))

Test accuracy (chord): 0.12594439596060342
Test accuracy (duration): 0.2807380625857125


In [8]:
import numpy as np
import pandas as pd

chord_cols = clf_chord.feature_names_in_
dur_cols   = clf_duration.feature_names_in_

init_chord_ctx = list(Xc_test.iloc[0])
init_dur_ctx   = list(Xd_test.iloc[0])[: len(Xd_test.iloc[0]) - len(init_chord_ctx)]

steps      = 100
generated  = []
chord_ctx  = init_chord_ctx.copy()
dur_ctx    = init_dur_ctx.copy()

for step in range(steps):
    Xc_df = pd.DataFrame([chord_ctx], columns=chord_cols)
    proba_c = clf_chord.predict_proba(Xc_df)[0]
    enc_chord = np.random.choice(clf_chord.classes_, p=proba_c)
    
    chord_ctx = chord_ctx[1:] + [enc_chord]
    Xd_df = pd.DataFrame([chord_ctx + dur_ctx], columns=dur_cols)
    proba_d = clf_duration.predict_proba(Xd_df)[0]
    next_dur = np.random.choice(clf_duration.classes_, p=proba_d)
    dur_ctx = dur_ctx[1:] + [next_dur]
    
    generated.append({
        'chord':    enc_chord,
        'duration': next_dur
    })

df_gen = pd.DataFrame(generated)
df_gen['chord'] = le_chord.inverse_transform(df_gen['chord'].values)
df_gen['duration'] = le_dur.inverse_transform(df_gen['duration'].values)
df_gen.to_csv('generated_sequence_tree.csv', index=False)
print(f"\nWrote {len(df_gen)} events to generated_sequence_full.csv")


Wrote 100 events to generated_sequence_full.csv


In [9]:
clf = IncrementalRBFSVC(random_state=42, verbose=False)
# 2. Define the parameter grid
param_grid = {
    'gamma':  ['scale', 'auto'],
    'loss':   ['log', 'modified_huber', 'perceptron'],
    'C':      [0.1, 1, 10, 100],
    'chunk_size': [100, 1000, 5000, 10000, 50000, 100000]
}

halving_search = HalvingRandomSearchCV(
    estimator=clf,
    param_distributions=param_grid,
    resource='n_samples',      # train on subsets of increasing size
    max_resources='auto',
    factor=3,                  # cut ⅔ of candidates each rung
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
)
halving_search.fit(Xc_test, yc_test)
print("Best params:", halving_search.best_params_)
"""
# 6. Examine results
print("Best parameters found:", grid_search.best_params_)
print(f"Best cross-val accuracy: {grid_search.best_score_:.3f}")

# 7. Evaluate on the hold-out test set
best_svc = grid_search.best_estimator_
y_pred = best_svc.predict(X_test)
print("\nTest set performance:")
print(classification_report(y_test, y_pred))
"""

NameError: name 'IncrementalRBFSVC' is not defined